# 00 - Understand the MOBPRO dictionary

**Source:** `varmod_mobpro_2023.csv` (full file).
**Module:** `scripts/datasets/mobpro_dictionary.py`.

Start here to understand the variable names and transport/geography codes. This notebook
explores only the dictionary. It does not load or join the individual commuting records.
All tables and plots describe dictionary entries, not people or journeys.


In [20]:
import sys
from pathlib import Path
import pandas as pd
import plotly.io as pio
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Open the hackathon project folder before running this notebook.')
if not Path(sys.executable).resolve().is_relative_to(ROOT / '.venv'):
    raise RuntimeError('Select the project .venv kernel, then restart and run all.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.datasets import mobpro_dictionary
from scripts.exploration import column_summary, repeated_keys, save_parquet

pio.renderers.default = 'plotly_mimetype'
print('Python:', sys.executable)


Python: d:\Hackathon Mobilités 2026 Spécial Dataviz\.venv\Scripts\python.exe


In [21]:
source = ROOT / mobpro_dictionary.FILENAME
raw = mobpro_dictionary.load(source)


In [22]:
print('Print rows and columns--shape:', raw.shape)
display(raw.head(10))
raw.info()


Print rows and columns--shape: (70627, 6)


,COD_VAR,LIB_VAR,COD_MOD,LIB_MOD,TYPE_VAR,LONG_VAR
0,COMMUNE,Département et commune du lieu de résidence,01001,L'Abergement-Clémenciat,CHAR,5
1,COMMUNE,Département et commune du lieu de résidence,01002,L'Abergement-de-Varey,CHAR,5
2,COMMUNE,Département et commune du lieu de résidence,01004,Ambérieu-en-Bugey,CHAR,5
3,COMMUNE,Département et commune du lieu de résidence,01005,Ambérieux-en-Dombes,CHAR,5
4,COMMUNE,Département et commune du lieu de résidence,01006,Ambléon,CHAR,5
5,COMMUNE,Département et commune du lieu de résidence,01007,Ambronay,CHAR,5
6,COMMUNE,Département et commune du lieu de résidence,01008,Ambutrix,CHAR,5
7,COMMUNE,Département et commune du lieu de résidence,01009,Andert-et-Condon,CHAR,5
8,COMMUNE,Département et commune du lieu de résidence,01010,Anglefort,CHAR,5
9,COMMUNE,Département et commune du lieu de résidence,01011,Apremont,CHAR,5


<class 'pandas.DataFrame'>
RangeIndex: 70627 entries, 0 to 70626
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   COD_VAR   70627 non-null  string
 1   LIB_VAR   70627 non-null  string
 2   COD_MOD   70627 non-null  string
 3   LIB_MOD   70627 non-null  string
 4   TYPE_VAR  70627 non-null  string
 5   LONG_VAR  70627 non-null  string
dtypes: string(6)
memory usage: 9.6 MB


## Clean a copy and inspect the columns

The original file and `raw` table stay unchanged. Strip surrounding spaces and turn
blank cells into missing values; preserve special codes such as `NA` and `ZZ`.
Dataset-specific numeric conversions are documented in the module's `clean()` function.
Unexpected numeric text raises an error for inspection. No rows are dropped or deduplicated.


In [23]:
cleaned = mobpro_dictionary.clean(raw)
assert len(cleaned) == len(raw)
display(cleaned.head())
display(column_summary(cleaned))


,COD_VAR,LIB_VAR,COD_MOD,LIB_MOD,TYPE_VAR,LONG_VAR
0,COMMUNE,Département et commune du lieu de résidence,01001,L'Abergement-Clémenciat,CHAR,5
1,COMMUNE,Département et commune du lieu de résidence,01002,L'Abergement-de-Varey,CHAR,5
2,COMMUNE,Département et commune du lieu de résidence,01004,Ambérieu-en-Bugey,CHAR,5
3,COMMUNE,Département et commune du lieu de résidence,01005,Ambérieux-en-Dombes,CHAR,5
4,COMMUNE,Département et commune du lieu de résidence,01006,Ambléon,CHAR,5


,data_type,missing_rows,distinct_values
column,,,
COD_VAR,string,0,32
LIB_VAR,string,0,32
COD_MOD,string,0,36485
LIB_MOD,string,0,34313
TYPE_VAR,string,0,2
LONG_VAR,string,0,5


## Analyse definitions and listed codes

Some entries describe numeric variables rather than category lists. `listed_codes` counts
literal codes in the dictionary, including `NA` where the source supplies that token.


In [24]:
variable_summary = mobpro_dictionary.variables(cleaned)
display(variable_summary)

VARIABLE = 'TRANS'  # Try IPONDI, COMMUNE, DCLT, REGION, REGLT or ILT.
display(mobpro_dictionary.lookup(cleaned, VARIABLE))
display(repeated_keys(cleaned, ['COD_VAR', 'COD_MOD']))


,COD_VAR,definition,data_type,listed_codes
0,AGEREVQ,Âge quinquennal en années révolues,CHAR,21
1,ARM,"Arrondissement municipal de résidence (Paris, ...",CHAR,46
2,COMMUNE,Département et commune du lieu de résidence,CHAR,34768
3,DCFLT,Commune frontalière du lieu de travail,CHAR,1517
4,DCLT,"Département, commune et arrondissement municip...",CHAR,34073
5,DEROU,Nombre de deux-roues à moteur du ménage (DOM),CHAR,6
6,DIPL,Diplôme le plus élevé,CHAR,10
7,EMPL,Condition d'emploi,CHAR,3
8,GS,Groupe socioprofessionnel en 6 postes,CHAR,6
9,ILT,Indicateur du lieu de travail,CHAR,7


,COD_VAR,LIB_VAR,COD_MOD,LIB_MOD,TYPE_VAR,LONG_VAR
0,TRANS,Mode de transport principal le plus souvent ut...,1,Pas de transport,CHAR,1
1,TRANS,Mode de transport principal le plus souvent ut...,2,"Marche à pied (ou rollers, patinette)",CHAR,1
2,TRANS,Mode de transport principal le plus souvent ut...,3,Vélo (y compris à assistance électrique),CHAR,1
3,TRANS,Mode de transport principal le plus souvent ut...,4,Deux-roues motorisé,CHAR,1
4,TRANS,Mode de transport principal le plus souvent ut...,5,"Voiture, camion, fourgonnette",CHAR,1
5,TRANS,Mode de transport principal le plus souvent ut...,6,Transports en commun,CHAR,1


,COD_VAR,COD_MOD,records


In [28]:
figure = mobpro_dictionary.plot_variables(variable_summary, top_n=10)
figure.show()


## Questions to answer before leaving this notebook

- What does an individual weight mean, and how is it different from a row count?
- What are the six usual main commuting modes?
- What is the difference between residence and workplace geography?
- Which values are special codes rather than blanks?

The variable-summary table above is the text alternative to the chart.


## Optional: save this dataset as Parquet



In [29]:
SAVE_CLEANED = False
output_path = ROOT / 'data' / 'cleaned' / 'mobpro_dictionary.parquet'

if SAVE_CLEANED:
    save_parquet(cleaned, output_path)
    print('Saved', len(cleaned), 'rows to', output_path)
else:
    print('No cleaned data written. Change SAVE_CLEANED after reviewing this dataset.')


No cleaned data written. Change SAVE_CLEANED after reviewing this dataset.
